In this notebook, we will show how to conduct co-simulation for an air-cooled data center with one CRAC unit and a chiller plant. In this case, the control variables are the the CRAC supply air flow rate and the supply air temperature setpoint.

In [ ]:
import pickle

import numpy as np

from dctwin.interfaces.gym_env import CoSimEnv
from dctwin.utils import config as env_config
from dctwin.utils import read_engine_config, setup_logging

from google.protobuf import json_format

# with open("model/POD/BC.pkl", "rb") as f:
#     bc = pickle.load(f)
# with open("model/POD/mean temperature field.pkl", "rb") as f:
#     mean_obs = pickle.load(f)
# with open("model/POD/POD Coefficients Train.pkl", "rb") as f:
#     coef = pickle.load(f)
# with open("model/POD/pod modes.pkl", "rb") as f:
#     modes = pickle.load(f)
#
# data = {
#     "mean_obs": mean_obs,
#     "modes": modes,
#     "train_bc": bc,
#     "train_coef": coef
# }
# with open("model/POD/data.pkl", "wb") as f:
#     pickle.dump(data, f)

(1) Setup environment variables

In [ ]:
engine_config = "config.prototxt"
env_config.eplus.engine_config_file = engine_config

(2) Read configuration files

In [ ]:
config = read_engine_config(engine_config=engine_config)
setup_logging(config.logging_config, engine_config=engine_config)

(3) Build environment

In [ ]:
env_config_name = config.WhichOneof("EnvConfig")
env_params = json_format.MessageToDict(
    getattr(config, env_config_name).env_params,
    preserving_proto_field_name=True,
)
env = CoSimEnv(
    config=getattr(config, env_config_name),
    reward_fn=None,
    schedule_fn=None,
    **env_params,
)

(4) Run EnergyPlus-CFD/POD co-simulation

In [ ]:
air_supply_sp = 20.0
air_supply_flowrate = 15.0
env.reset()
done = False
while not done:
    act = np.array([air_supply_sp, air_supply_flowrate])
    obs, rew, done, truncated, info = env.step(act)